<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# Chapter 4 Exercise solutions

# Exercise 4.1: Parameters in the feed forward versus attention module

In [3]:
# 导入第 4 章已经实现好的 Transformer 块
from gpt import TransformerBlock

# GPT-2 small（约 1.24 亿个非重复参数）的基础配置
GPT_CONFIG_124M = {
    "vocab_size": 50257,      # 词表中可识别的 token 总数
    "context_length": 1024,  # 模型一次最多处理的 token 数
    "emb_dim": 768,          # 每个 token 的向量维度
    "n_heads": 12,           # 多头注意力中的注意力头数量
    "n_layers": 12,          # 完整 GPT 模型包含的 Transformer 块数量
    "drop_rate": 0.1,        # 训练时随机丢弃 10% 的中间结果，缓解过拟合
    "qkv_bias": False        # Q、K、V 三个线性投影不使用偏置项
}

# 本练习只创建一个 Transformer 块，用来比较其内部两个子模块的参数量
block = TransformerBlock(GPT_CONFIG_124M)

In [4]:
# p.numel() 返回一个参数张量中的元素个数；求和得到前馈网络的全部参数量
total_params = sum(p.numel() for p in block.ff.parameters())
# :, 让整数按千位分隔显示，便于阅读
print(f"Total number of parameters in feed forward module: {total_params:,}")

Total number of parameters in feed forward module: 4,722,432


In [5]:
# 同样统计多头注意力模块中所有可学习参数的数量
total_params = sum(p.numel() for p in block.att.parameters())
# 结果可与上一个单元格的前馈网络参数量直接比较
print(f"Total number of parameters in attention module: {total_params:,}")

Total number of parameters in attention module: 2,360,064


- The results above are for a single transformer block
- Optionally multiply by 12 to capture all transformer blocks in the 124M GPT model

# Exercise 4.2: Initialize larger GPT models

- **GPT2-small** (the 124M configuration we already implemented):
    - "emb_dim" = 768
    - "n_layers" = 12
    - "n_heads" = 12

- **GPT2-medium:**
    - "emb_dim" = 1024
    - "n_layers" = 24
    - "n_heads" = 16

- **GPT2-large:**
    - "emb_dim" = 1280
    - "n_layers" = 36
    - "n_heads" = 20

- **GPT2-XL:**
    - "emb_dim" = 1600
    - "n_layers" = 48
    - "n_heads" = 25

In [6]:
# 先准备一份通用基础配置；不同规模只修改模型宽度、深度和注意力头数
GPT_CONFIG_124M = {
    "vocab_size": 50257,      # 四种 GPT-2 共用同一套词表
    "context_length": 1024,  # 四种模型的最大上下文长度均为 1024
    "emb_dim": 768,          # 默认使用 small 的嵌入维度
    "n_heads": 12,           # 默认使用 small 的注意力头数
    "n_layers": 12,          # 默认使用 small 的 Transformer 块数量
    "drop_rate": 0.1,        # 各处统一使用的 dropout 概率
    "qkv_bias": False        # Q、K、V 投影不使用偏置项
}


def get_config(base_config, model_name="gpt2-small"):
    """根据模型名称返回对应的 GPT-2 配置字典。"""
    # 使用浅拷贝，避免修改传入的基础配置，方便循环中反复复用
    GPT_CONFIG = base_config.copy()

    # emb_dim 决定模型宽度，n_layers 决定深度，n_heads 决定并行注意力头数
    if model_name == "gpt2-small":
        GPT_CONFIG["emb_dim"] = 768
        GPT_CONFIG["n_layers"] = 12
        GPT_CONFIG["n_heads"] = 12

    elif model_name == "gpt2-medium":
        GPT_CONFIG["emb_dim"] = 1024
        GPT_CONFIG["n_layers"] = 24
        GPT_CONFIG["n_heads"] = 16

    elif model_name == "gpt2-large":
        GPT_CONFIG["emb_dim"] = 1280
        GPT_CONFIG["n_layers"] = 36
        GPT_CONFIG["n_heads"] = 20

    elif model_name == "gpt2-xl":
        GPT_CONFIG["emb_dim"] = 1600
        GPT_CONFIG["n_layers"] = 48
        GPT_CONFIG["n_heads"] = 25

    else:
        # 对不支持的名称立即报错，避免悄悄使用错误配置
        raise ValueError(f"Incorrect model name {model_name}")

    return GPT_CONFIG


def calculate_size(model):
    """打印模型参数量，并粗略估算 float32 参数占用的内存。"""
    # 统计当前模型对象实际保存的全部参数（包括独立的输出层）
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total number of parameters: {total_params:,}")

    # 原版 GPT-2 让输出层和 token 嵌入层共享权重（weight tying），不必保存两份矩阵
    # 此处通过减去输出层参数做理论估算；当前模型代码并未真正绑定这两个权重
    total_params_gpt2 = total_params - sum(p.numel() for p in model.out_head.parameters())
    print(f"Number of trainable parameters considering weight tying: {total_params_gpt2:,}")

    # 假设每个参数都是 float32，即每个参数占 4 字节
    # 这里只计算参数本身，不包含梯度、优化器状态和前向传播的中间激活值
    total_size_bytes = total_params * 4

    # 将字节换算为 MiB（代码沿用书中的 MB 显示名称）
    total_size_mb = total_size_bytes / (1024 * 1024)

    print(f"Total size of the model: {total_size_mb:.2f} MB")

In [7]:
# 导入第 4 章实现的完整 GPT 模型
from gpt import GPTModel


# 依次创建 small、medium、large 和 xl 四种规模的模型
for model_abbrev in ("small", "medium", "large", "xl"):
    # 例如把 "small" 拼成 get_config 能识别的 "gpt2-small"
    model_name = f"gpt2-{model_abbrev}"
    CONFIG = get_config(GPT_CONFIG_124M, model_name=model_name)
    # 根据当前配置实例化模型；这里只创建模型，还没有训练或推理
    model = GPTModel(CONFIG)
    print(f"\n\n{model_name}:")
    # 输出当前规模的参数量与理论内存占用
    calculate_size(model)



gpt2-small:
Total number of parameters: 163,009,536
Number of trainable parameters considering weight tying: 124,412,160
Total size of the model: 621.83 MB


gpt2-medium:
Total number of parameters: 406,212,608
Number of trainable parameters considering weight tying: 354,749,440
Total size of the model: 1549.58 MB


gpt2-large:
Total number of parameters: 838,220,800
Number of trainable parameters considering weight tying: 773,891,840
Total size of the model: 3197.56 MB


gpt2-xl:
Total number of parameters: 1,637,792,000
Number of trainable parameters considering weight tying: 1,557,380,800
Total size of the model: 6247.68 MB


# Exercise 4.3: Using separate dropout parameters

In [8]:
# 将原来的统一 drop_rate 拆成三个独立超参数，以便分别调节不同位置的正则化强度
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate_emb": 0.1,       # token 嵌入与位置嵌入相加后的 dropout
    "drop_rate_attn": 0.1,      # 多头注意力内部注意力权重的 dropout
    "drop_rate_shortcut": 0.1,  # 注意力/前馈输出加入残差前的 dropout
    "qkv_bias": False
}

In [9]:
import torch.nn as nn
from gpt import MultiHeadAttention, LayerNorm, FeedForward


class TransformerBlock(nn.Module):
    """由注意力子层和前馈子层组成的 Transformer 块。"""

    def __init__(self, cfg):
        super().__init__()
        # 自注意力让每个 token 根据前文信息更新自己的表示
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            # 只把注意力专用的 dropout 概率传给注意力模块
            dropout=cfg["drop_rate_attn"],
            qkv_bias=cfg["qkv_bias"])
        # 前馈网络会把每个 token 的向量先扩展到 4 倍维度，再压回原维度
        self.ff = FeedForward(cfg)
        # 两个子层各自使用一次层归一化（Pre-LayerNorm 结构）
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        # 注意力或前馈结果与原输入相加之前，共用这一 dropout 配置
        self.drop_shortcut = nn.Dropout(cfg["drop_rate_shortcut"])

    def forward(self, x):
        # 第一段：层归一化 -> 多头注意力 -> dropout -> 残差相加
        shortcut = x                 # 暂存原输入，供残差连接使用
        x = self.norm1(x)
        x = self.att(x)              # 形状仍为 [批量大小, token 数, 嵌入维度]
        x = self.drop_shortcut(x)
        x = x + shortcut             # 加回原输入，帮助深层网络稳定传递信息和梯度

        # 第二段：层归一化 -> 前馈网络 -> dropout -> 残差相加
        shortcut = x                 # 此处暂存的是注意力子层处理后的结果
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut

        return x


class GPTModel(nn.Module):
    """使用三类独立 dropout 的 GPT 模型。"""

    def __init__(self, cfg):
        super().__init__()
        # token 嵌入：把离散 token ID 转换为可学习的连续向量
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        # 位置嵌入：告诉模型每个 token 位于序列中的哪个位置
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        # 嵌入层使用自己的 dropout 概率
        self.drop_emb = nn.Dropout(cfg["drop_rate_emb"])

        # 按 n_layers 指定的数量堆叠 Transformer 块
        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        # 所有 Transformer 块之后再做一次归一化
        self.final_norm = LayerNorm(cfg["emb_dim"])
        # 将每个位置的隐藏向量映射成词表中每个 token 的预测分数
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)

    def forward(self, in_idx):
        # in_idx 的形状是 [批量大小, 序列长度]，元素为 token ID
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        # 生成 0 到 seq_len-1 的位置编号，并放到与输入相同的设备上
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        # token 信息与位置信息相加；形状为 [批量大小, token 数, 嵌入维度]
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)          # 只在嵌入结果处使用 embedding dropout
        x = self.trf_blocks(x)        # 依次通过全部 Transformer 块
        x = self.final_norm(x)
        # logits 形状为 [批量大小, token 数, 词表大小]
        logits = self.out_head(x)
        return logits

In [10]:
# forward 方法会用到 torch.arange，因此在真正执行前向传播前导入 torch
import torch

# 固定随机种子，使模型的随机初始化在重复运行时保持一致，便于复现实验
torch.manual_seed(123)
# 根据拆分后的 dropout 配置实例化模型；此处尚未执行训练或推理
model = GPTModel(GPT_CONFIG_124M)